 Live survey response analysis

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display

from survey_responce import (
	limit_response_cases,
	rank_borda,
	rank_by_category,
	rank_overall,
	rank_plackett_luce,
	summarize_participants,
)

In [ ]:
config_path = "live-analisys-config.json"
survey_path = "../eval/output/survey.json"

with open(config_path, encoding="utf-8") as config_file:
	config = json.load(config_file)


def fetch_api(path):
	request = Request(
		f"{config['base_url'].rstrip('/')}{path}",
		headers={"Authorization": f"Bearer {config['admin_token']}"},
	)
	with urlopen(request, timeout=30) as api_response:
		return json.load(api_response)


n_cases = 99
responses = fetch_api("/api/responses")
response_dir = Path("..") / "eval" / "final-assesment"
response_dir.mkdir(parents=True, exist_ok=True)
response_path = response_dir / f"responce-{datetime.now():%y-%m-%d-%H-%M}.json"
with response_path.open("w", encoding="utf-8") as response_file:
	json.dump(responses, response_file, ensure_ascii=False, indent=2)
response_labels = pd.Series([response.get("label") for response in responses], dtype="string")
responses = [
	response
	for response, include in zip(responses, response_labels.str.match(r"^p\d{2}", na=False))
	if include
]
responses = limit_response_cases(responses, survey_path, n_cases)
participants = fetch_api("/api/participants/")


 Responses by participant

In [ ]:
participant_stats = pd.DataFrame(
	summarize_participants(responses, participants, survey_path),
	columns=[
		"participant", "group", "status", "saved_at",
		"completed_cases", "incomplete_cases", "complete_rankings", "last_page",
	],
)
participant_stats["group"] = pd.to_numeric(participant_stats["group"]).astype("Int64")
participant_stats = participant_stats.astype({"last_page": "Int64"})
status_counts = participant_stats["status"].value_counts()
display(Markdown(
	f"**{len(participant_stats)}** participants with responses: "
	f"**{status_counts.get('submitted', 0)}** submitted, **{status_counts.get('draft', 0)}** drafts; "
	f"**{participant_stats['completed_cases'].sum()}** completed cases and "
	f"**{participant_stats['complete_rankings'].sum()}** complete rankings in total."
))

participant_stats.assign(
    saved_at=pd.to_datetime(participant_stats["saved_at"], utc=True)
                .dt.tz_convert("Europe/Belgrade")
                .dt.strftime("%Y-%m-%d %H:%M")
).sort_values(
    ["saved_at", "status", "participant"], ascending=[False,False, True], ignore_index=True
)



Rankings by category

In [ ]:
from IPython.display import HTML
from survey_responce import extract_triplets_by_case_type

triplets_by_case_type = extract_triplets_by_case_type(
	responses, survey_path, "../eval/test-cases.yml", skip_incomplete=True,
)
triplets_by_category = triplets_by_case_type["total"]
rank_method = rank_borda
# rank_method = rank_plackett_luce


def ranking_dataset(rankings_by_type):
	frames = [
		pd.DataFrame(rankings_by_type[case_type], columns=["model", "rank", "score", "answer_count"])
		.set_index("model").add_suffix(f"_{case_type}")
		for case_type in ("total", "inclusive", "general")
	]
	dataset = pd.concat(frames, axis=1).reset_index()
	for case_type in rankings_by_type:
		dataset[f"rank_{case_type}"] = dataset[f"rank_{case_type}"].astype("Int64")
		dataset[f"answer_count_{case_type}"] = dataset[f"answer_count_{case_type}"].fillna(0).astype("Int64")
	return dataset.sort_values("rank_total").set_index("rank_total")


def display_ranking_tables(dataset):
	tables = []
	for case_type in ("total", "inclusive", "general"):
		table = dataset.reset_index()[[
			f"rank_{case_type}", "model", f"score_{case_type}", f"answer_count_{case_type}",
		]].rename(columns={
			f"rank_{case_type}": "rank",
			f"score_{case_type}": "score",
			f"answer_count_{case_type}": "answer_count",
		}).dropna(subset=["rank"]).sort_values("rank").set_index("rank")
		tables.append(
			f'<section style="flex: 0 0 auto;"><h4>{case_type.title()}</h4>'
			f'{table.to_html(border=0)}</section>'
		)
	display(HTML(
		'<div style="display: flex; gap: 24px; overflow-x: auto; align-items: flex-start;">'
		+ "".join(tables) + "</div>"
	))


category_rankings_by_type = {
	case_type: rank_by_category(triplets, rank_method)
	for case_type, triplets in triplets_by_case_type.items()
}
rankings_by_category = {
	category: ranking_dataset({
		case_type: categories.get(category, [])
		for case_type, categories in category_rankings_by_type.items()
	})
	for category in triplets_by_category
}
for category, dataset in rankings_by_category.items():
	display(Markdown(f"### {category}"))
	display_ranking_tables(dataset)

overall_rankings = ranking_dataset({
	case_type: rank_overall(triplets, rank_method)
	for case_type, triplets in triplets_by_case_type.items()
})
if triplets_by_category:
	display(Markdown("### Overall"))
	display_ranking_tables(overall_rankings)
else:
	display(Markdown("No complete rankings yet."))

### Participant question timeline

First recorded appearance of each question in saved answers (UTC), not completion time. This timeline includes all recorded questions, independently of `n_cases`.

In [ ]:
timeline_participant = "p08"
timeline_matches = [participant for participant in participants if participant["label"] == timeline_participant]
if len(timeline_matches) != 1:
	raise ValueError(f"Expected one participant labelled {timeline_participant!r}; found {len(timeline_matches)}.")

participant_questions = fetch_api("/api/participant-questions")
question_timeline = pd.DataFrame(
	[
		{"question_name": question["question_name"], "first_answered_at": question["first_answered_at"]}
		for question in participant_questions
		if question["token"] == timeline_matches[0]["token"]
	],
	columns=["question_name", "first_answered_at"],
)
question_timeline["first_answered_at"] = pd.to_datetime(question_timeline["first_answered_at"], format="ISO8601", utc=True, errors="raise")
question_timeline = question_timeline.sort_values("first_answered_at", kind="stable", ignore_index=True)
display(Markdown(f"### Question timeline: {timeline_participant}"))
if question_timeline.empty:
	display(Markdown("No question timestamps recorded for this participant."))
else:
	with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
		display(question_timeline.assign(first_answered_at=question_timeline["first_answered_at"]
		                                 .dt.tz_convert("Europe/Belgrade")
		                                 .dt.strftime("%Y-%m-%d %H:%M")))
